# Select items for an annotation study

Run the form cell, set your parameters, then run the Python cells below in order. Use **Synthetic pilot** to try sampling with the bundled sample tables. To preview a synthetic draft in the Hub, the service must be using those same tables. Use **Seminar corpus** for the classroom studies.

The sampling unit is one row of the source table, a post text, a comment or an image. Comment drafts require an approved comments snapshot in the service. Image drafts can use any row of `images.csv` (**Item type** `Cover`, `Gallery` or `Frame`), provided the image file lies in the raw or derived corpus. The service needs the approved `images.csv` snapshot. Keep outputs containing corpus material inside your private workspace.

## Classroom coding rounds

Choose **Seminar corpus → Post text** and enter `Caption` in **Item type**. One item is one TikTok caption with its own `Text ID`.

- **30 Nov:** Draw a small shared pilot sample. At least two coders independently apply the draft codebook to each caption.
- **7 Dec:** Draw a fresh caption sample to try the revised codebook, then freeze it for the test round.
- **14 Dec:** Draw a held-out caption sample, excluding posts used in either pilot or for codebook development. At least two coders independently code each caption using the frozen codebook.

Keep each round's draft and seed. Put earlier caption IDs in **Exclude these IDs** when drawing a new sample, and use a new study ID when you publish.

In [ ]:
import json
import os
import random
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

table_directories = {
    "Synthetic pilot": Path("annotation_pilot"),
    "Seminar corpus": Path(os.environ.get("SEMINAR_OUTPUT", "/corpus/derived")) / "tables",
}
sources = {
    "texts": ("Text ID", "Text Type"),
    "comments": ("Comment ID", None),
    "images": ("Image ID", "Image Type"),
}
style = {"description_width": "initial"}

selection_fields = {
    "dataset": widgets.Dropdown(options=list(table_directories), description="Data", style=style),
    "source": widgets.Dropdown(
        options=[("Post text", "texts"), ("Comments", "comments"), ("Images", "images")],
        description="Source", style=style,
    ),
    "label": widgets.Text(description="Collection label", placeholder="Any", style=style),
    "strategy": widgets.Dropdown(
        options=[("Any", ""), ("FYP", "FYP"), ("search", "search"),
                 ("hashtag", "hashtag"), ("profile", "profile")],
        description="Collection strategy", style=style,
    ),
    "item_type": widgets.Text(description="Item type", placeholder="Any", style=style),
    "search": widgets.Text(description="Text contains", style=style),
    "include": widgets.Textarea(description="Include only these IDs", placeholder="One ID per line", rows=2, style=style),
    "exclude": widgets.Textarea(description="Exclude these IDs", placeholder="One ID per line", rows=2, style=style),
    "sampling": widgets.Dropdown(
        options=[("All matching items", "all"), ("Simple random sample", "random")],
        value="random", description="Sampling", style=style,
    ),
    "size": widgets.IntText(value=2, description="Sample size", style=style),
    "seed": widgets.IntText(value=42, description="Seed", style=style),
}
title = widgets.Text(description="Working title", style=style)
filename = widgets.Text(value="study-draft.json", description="Draft filename", style=style)
preview_parameters = None
display(widgets.VBox([title, *selection_fields.values(), filename]))

## 1. Load the corpus table

Read the current widget values into an ordinary Python dictionary. Loading IDs as strings preserves leading zeroes, including those in the synthetic pilot. Inspect the rows and column names before deciding which filters to apply.

In [ ]:
preview_parameters = None
eligible = None
parameters = {name: field.value for name, field in selection_fields.items()}
source = parameters["source"]
tables_dir = table_directories[parameters["dataset"]]
id_column, type_column = sources[source]
source_path = tables_dir / f"{source}.csv"

if not source_path.is_file():
    raise FileNotFoundError(f"{source_path} is unavailable. Choose a source with a table.")
items = pd.read_csv(source_path, dtype=str, keep_default_na=False)
if items[id_column].str.strip().eq("").any() or items[id_column].duplicated().any():
    raise ValueError(f"{source_path.name} must contain one row per non-empty {id_column}.")

print(f"Loaded {len(items)} rows.")
display(items.head())

## 2. Define the eligible population

Collection filters select posts by matching label and strategy on the same membership row. The other filters operate on the items themselves. Add any project-specific pandas filters here, before drawing the sample.

In [ ]:
preview_parameters = None
eligible = items.copy()

if parameters["label"].strip() or parameters["strategy"]:
    memberships = pd.read_csv(tables_dir / "collections.csv", dtype=str, keep_default_na=False)
    if parameters["label"].strip():
        memberships = memberships[memberships["label"] == parameters["label"].strip()]
    if parameters["strategy"]:
        memberships = memberships[memberships["strategy"] == parameters["strategy"]]
    eligible = eligible[eligible["Post ID"].isin(memberships["Post ID"])]

if parameters["item_type"].strip():
    if type_column is None:
        raise ValueError("Item type is available for post text and images.")
    eligible = eligible[eligible[type_column] == parameters["item_type"].strip()]

if parameters["search"]:
    if "Text" not in eligible:
        raise ValueError("Text search is available for post text and comments.")
    eligible = eligible[eligible["Text"].str.contains(parameters["search"], case=False, regex=False)]

# Add project-specific pandas filters here.

include_ids = [line.strip() for line in parameters["include"].splitlines() if line.strip()]
exclude_ids = [line.strip() for line in parameters["exclude"].splitlines() if line.strip()]
if include_ids:
    if len(include_ids) != len(set(include_ids)):
        raise ValueError("Enter each included ID only once.")
    missing = set(include_ids).difference(eligible[id_column])
    if missing:
        raise ValueError(f"These included IDs are missing from the filtered rows: {sorted(missing)[:8]}")
    eligible = eligible.set_index(id_column, drop=False).loc[include_ids]
eligible = eligible[~eligible[id_column].isin(exclude_ids)]

print(f"{len(eligible)} eligible items after filtering.")

## 3. Draw and inspect the sample

A simple random sample draws the requested number of distinct items from the eligible population. Sorting the IDs before drawing makes the result reproducible for the same seed and eligible IDs, even if the CSV row order changes. **All matching items** keeps the eligible rows in order and ignores the sample size and seed.

In [ ]:
preview_parameters = None
if eligible is None:
    raise ValueError("Run the filter cell before drawing a sample.")
if eligible.empty:
    raise ValueError("No items match these filters.")

if parameters["sampling"] == "random":
    sample_size = parameters["size"]
    if not 1 <= sample_size <= len(eligible):
        raise ValueError(f"Choose a positive sample size of at most {len(eligible)} eligible items.")
    rng = random.Random(parameters["seed"])
    sampled_ids = rng.sample(sorted(eligible[id_column]), sample_size)
    selected = eligible.set_index(id_column, drop=False).loc[sampled_ids].reset_index(drop=True)
    print(f"Random sample of {len(selected)} of {len(eligible)} eligible items (seed {parameters['seed']}).")
else:
    selected = eligible.reset_index(drop=True)
    print(f"Selected all {len(selected)} eligible items.")

preview_parameters = parameters.copy()
preview_ids = selected[id_column].tolist()
display(selected.head(25))

## 4. Export the reviewed draft

Once the preview looks right, run this cell. If you change any selection inputs in the form, rerun the load, filter, and sample cells before exporting. The JSON contains the title, source table, and ordered IDs. Download it from `annotation-drafts` in the JupyterLab file browser. Open **Services → Annotation Studies → Create study**, upload the JSON under **Notebook draft**, write your instructions and questions, and click **Preview coding form**. Check the items and their order before you click **Publish to class**.

In [ ]:
if preview_parameters is None:
    raise ValueError("Run the load, filter, and sample cells before exporting.")
if preview_parameters != {name: field.value for name, field in selection_fields.items()}:
    raise ValueError("Selection inputs changed. Rerun the load, filter, and sample cells before exporting.")

working_title = title.value.strip()
if not 1 <= len(working_title) <= 200:
    raise ValueError("Enter a working title of up to 200 characters.")
draft_name = filename.value.strip()
if Path(draft_name).name != draft_name or Path(draft_name).suffix != ".json":
    raise ValueError("Enter a draft filename ending in .json.")

draft_dir = Path.home() / "annotation-drafts"
draft_dir.mkdir(mode=0o700, parents=True, exist_ok=True)
draft_path = draft_dir / draft_name
draft = {
    "version": 1,
    "title": working_title,
    "source": preview_parameters["source"],
    "item_ids": preview_ids,
}

# Exclusive creation protects any draft saved under the same name.
with draft_path.open("x", encoding="utf-8") as output:
    json.dump(draft, output, ensure_ascii=False, indent=2)
    output.write("\n")
print(f"Saved {len(draft['item_ids'])} item IDs to {draft_path}.")